# Purpose of Script

The purpose of this script is to simply get familiar with Google Cloud, the formatting, and general data models.

Resource https://omophub.com/blog/omop-data-model$0

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use("ggplot")

from google.colab.sql import bigquery as bq

Influenza SNOMED CT Codes found through the url above

In [ ]:
## Stroke: https://evsexplore.semantics.cancer.gov/evsexplore/concept/snomedct_us/230690007$0
## Cerebrovascular accident ( Code - 230690007 )

## Heart attack: https://evsexplore.semantics.cancer.gov/evsexplore/concept/snomedct_us/22298006$0
## Myocardial infarction ( Code - 22298006 )

concept_codes = {'cerebro_accident': '230690007',
                  'myocardial_infarction': '22298006'}


concept_ids = {'cerebro_accident': 381316,
                'myocardial_infarction': 4329847}

## concept_code -> concept_id (lookup through concept table by searching concept_code)
## manually pasted above just for easy / explicit viewing

### For the tables viewed below, the concept 'code' as defined by the government website is used to query the `concept` table, and then afterwards, not used again.

In [ ]:
### beter direct in python
query = f"""
            SELECT *
            FROM `bigquery-public-data.cms_synthetic_patient_data_omop.concept`
            WHERE 1=1
            AND `vocabulary_id` = 'SNOMED'
            AND `concept_code` in {tuple(list(concept_codes.values()))}
            LIMIT 10
            ;"""

df = bq.run(query)
display(df)

In [ ]:
df.to_csv("test.csv", index=False)

## Condition Occurrence Preview

CONDITION_OCCURRENCE: This is where you’ll find all patient diagnoses, from chronic illnesses to acute infections. Each row represents a specific condition like 'Type 2 diabetes' or 'acute bronchitis,' tied to a patient and, usually, to the visit when it was first diagnosed.

In [ ]:
# query = """
#           SELECT *
#         FROM `bigquery-public-data.cms_synthetic_patient_data_omop.condition_occurrence`
#         limit 5
#         """

# df = bq.run(query)
# display(df)

query = f"""
        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.condition_occurrence`
        WHERE 1=1
        AND
        `condition_concept_id` in {tuple(concept_ids.values())} -- may be too low on the tree
        """

df = bq.run(query)
display(df)

# query = f"""
#         SELECT *
#         FROM `bigquery-public-data.cms_synthetic_patient_data_omop.condition_occurrence`
#         WHERE 1=1
#         AND `condition_concept_id` = 230690007
#         limit 5
#         """

# df = bq.run(query)
# display(df)

## Domain Preview

## Concept Relationships

In [ ]:
print(f"From government website (concept 'codes'):")
print(concept_codes)
print("\nFrom OMOP concept table (concept 'ids'):")
print(concept_ids)

In [ ]:
query = f"""
        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.concept_relationship`
        WHERE 1 = 1
        AND `concept_id_1` IN {tuple(concept_ids.values())}
        ;"""

df = bq.run(query)

## all "Condition" in domain_id - maybe always - maybe inconsistent
display(df)

## Concept Ancestors

In [ ]:
query = f"""
        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.concept_ancestor`
        WHERE 1 = 1
        AND `ancestor_concept_id` IN {tuple(concept_ids.values())}
        ;"""

df = bq.run(query)

## all "Condition" in domain_id - maybe always - maybe inconsistent
display(df)

In [ ]:
query = f"""

        WITH cardiac_subtypes AS
        ( SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.concept_ancestor`
        WHERE 1 = 1
        AND `ancestor_concept_id` IN {tuple(concept_ids.values())}
        ),

        disease_happened as (
        SELECT *, EXTRACT(MONTH from `condition_start_date`) AS condition_start_date_month
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.condition_occurrence`
        WHERE `condition_concept_id` IN
                    (
                    SELECT DISTINCT(f.`descendant_concept_id`)
                    FROM cardiac_subtypes f
                    )
        )

        select *
        FROM disease_happened
        """

df = bq.run(query)
display(df)

In [ ]:
month_order = df['condition_start_date_month'].value_counts().sort_index().reset_index()

fig, ax = plt.subplots(figsize=(10,4))

plt.bar(month_order['condition_start_date_month'], month_order['count'])
plt.xticks(np.arange(1, 13, 1))
plt.xlabel("Month")
plt.ylabel("Number of Stroke/Heart Attacks")
plt.title("Diagnoses per Month")

plt.show()

In [ ]:
## unique people and location

query = f"""

        WITH cardiac_subtypes AS
        ( SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.concept_ancestor`
        WHERE 1 = 1
        AND `ancestor_concept_id` IN {tuple(concept_ids.values())}
        ),

        diagnosis_happened as (
        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.condition_occurrence`
        WHERE `condition_concept_id` IN
                    (
                    SELECT DISTINCT(f.`descendant_concept_id`)
                    FROM cardiac_subtypes f
                    )
        ),

        people_w_diagnosis as (

        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.person`
        WHERE `person_id` IN
                    (
                    SELECT DISTINCT(f.`person_id`)
                    FROM diagnosis_happened f
                    )

        ),

        people_w_location as (
        SELECT *
        FROM people_w_diagnosis pf
        JOIN `bigquery-public-data.cms_synthetic_patient_data_omop.location` l
        using(`location_id`)
        )

        select *
        FROM people_w_location
        """

df = bq.run(query)
display(df)

In [ ]:
display(df['race_concept_id'].value_counts())

display(df['ethnicity_concept_id'].value_counts())

display(df['state'].value_counts().sort_values(ascending=False))

In [ ]:
query = f"""

        WITH cardiac_subtypes AS
        ( SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.concept_ancestor`
        WHERE 1 = 1
        AND `ancestor_concept_id` IN {tuple(concept_ids.values())}
        ),

        subtype_concept_details as (
        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.concept`
        WHERE `concept_id` IN
                    (
                    SELECT DISTINCT(f.`descendant_concept_id`)
                    FROM cardiac_subtypes f
                    )
        )

        select *
        FROM subtype_concept_details
        """

df = bq.run(query)
display(df)

In [ ]:
query = f"""

        WITH cardiac_subtypes AS
        ( SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.concept_ancestor`
        WHERE 1 = 1
        AND `ancestor_concept_id` IN {tuple(concept_ids.values())}
        ),

        diagnosis_happened as (
        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.condition_occurrence`
        WHERE `condition_concept_id` IN
                    (
                    SELECT DISTINCT(f.`descendant_concept_id`)
                    FROM cardiac_subtypes f
                    )
        )

        select COUNT(DISTINCT(`person_id`)) as `people_w_diagnosis`
        FROM diagnosis_happened
        """

df = bq.run(query)
display(df)

In [ ]:
query = f"""

        WITH cardiac_subtypes AS
        ( SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.concept_ancestor`
        WHERE 1 = 1
        AND `ancestor_concept_id` IN {tuple(concept_ids.values())}
        ),

        diagnosis_happened as (
        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.condition_occurrence`
        WHERE `condition_concept_id` IN
                    (
                    SELECT DISTINCT(f.`descendant_concept_id`)
                    FROM cardiac_subtypes f
                    )
        )

        select `person_id`, COUNT(`person_id`) as `num_times_w_cardiac_problem`
        FROM diagnosis_happened
        GROUP BY `person_id`
        ORDER BY num_times_w_cardiac_problem DESC
        """

df = bq.run(query)
display(df)

In [ ]:
df['num_times_w_cardiac_problem'].value_counts().sort_index(ascending=False)

In [ ]:
query = """
        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.condition_occurrence`
        WHERE `person_id` = 1137719
        ORDER BY `condition_start_date`
        """

df = bq.run(query)
display(df)


query = """
        WITH cardiac_often_person AS
        (SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.condition_occurrence`
        WHERE `person_id` = 1137719
        ),

        date_ranges as (
        SELECT min(condition_start_date) as `first_time_cardiac_problem`,
              max(condition_end_date) as `last_time_cardiac_problem`
              FROM cardiac_often_person f

        )

        SELECT *
        FROM date_ranges
        """

df = bq.run(query)
display(df)

In [ ]:
query  = """SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.death`
        LIMIT 100"""

df = bq.run(query)

display(df)


query  = """SELECT COUNT(DISTINCT(`person_id`)) as `num_deaths`
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.death`
        LIMIT 100"""

df = bq.run(query)

display(df)

In [ ]:
query = f"""

        WITH flu_subtypes AS
        ( SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.concept_ancestor`
        WHERE 1 = 1
        AND `ancestor_concept_id` IN {tuple(concept_ids.values())}
        ),

        death_from_flu as (
        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.death`
        WHERE `cause_concept_id` IN
                    (
                    SELECT DISTINCT(f.`descendant_concept_id`)
                    FROM flu_subtypes f
                    )
        OR `cause_concept_id` IN {tuple(concept_ids.values())}
        )

        select *
        FROM death_from_flu
        """

df = bq.run(query)
display(df)

In [ ]:
query = """
        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.person`
        WHERE `person_id` = 1137719
        """

df = bq.run(query)
display(df) ## year of birth 1919 ~ 107 years old

In [ ]:
query = """
        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.location`
        WHERE `location_id` = 1663
        """
df = bq.run(query)
display(df) # Missouri

## Miscellanous

In [ ]:
# bq.run("""SELECT
#     m.person_id,
#     m.measurement_date,
#     m.value_as_number
# FROM
#     `bigquery-public-data.cms_synthetic_patient_data_omop.measurement` m
# JOIN
#     `bigquery-public-data.cms_synthetic_patient_data_omop.concept` c

#     ON m.measurement_concept_id = c.concept_id
# WHERE
#     c.concept_name = 'Systolic blood pressure'
#     AND m.value_as_number > 140
#     AND m.measurement_date >= '2023-01-01';""")


# ## measurement table doesn't exist in the US?